In [ ]:
# Cell 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - Install libraries
!pip uninstall -y -q sentence-transformers
!pip install -q --force-reinstall --no-cache-dir --progress-bar off numpy==1.26.4
!pip install -q --progress-bar off transformers==4.40.0 scikit-learn==1.4.2 fairlearn==0.10.0 aif360==0.6.1 pandas==2.2.1 matplotlib==3.8.4 seaborn==0.13.2
!python -c "import numpy as np; import transformers; import sklearn; import fairlearn; print('numpy:', np.__version__); print('transformers:', transformers.__version__); print('sklearn:', sklearn.__version__); print('fairlearn:', fairlearn.__version__)"

In [ ]:
# Cell 3 - Unzip CSV files if needed
import os

zip_train = '/content/drive/MyDrive/jigsaw/jigsaw-unintended-bias-train.csv.zip'
zip_val = '/content/drive/MyDrive/jigsaw/validation.csv.zip'
out_dir = '/content/jigsaw/'

os.makedirs(out_dir, exist_ok=True)

train_csv = os.path.join(out_dir, 'jigsaw-unintended-bias-train.csv')
val_csv = os.path.join(out_dir, 'validation.csv')

if not (os.path.exists(train_csv) and os.path.exists(val_csv)):
    !unzip -q "{zip_train}" -d "{out_dir}"
    !unzip -q "{zip_val}" -d "{out_dir}"
else:
    print('Already extracted; skipping unzip.')

print('Files extracted:')
!ls /content/jigsaw/

In [ ]:
# Cell 4 - Set paths
DATA_PATH = '/content/jigsaw/jigsaw-unintended-bias-train.csv'
VAL_PATH = '/content/jigsaw/validation.csv'
BEST_MODEL_DIR = '/content/drive/MyDrive/jigsaw/best_mitigated_model'
DRIVE_OUT_DIR = '/content/drive/MyDrive/jigsaw'

# Part 5 - Production Guardrail Pipeline

This notebook is self-sustained and does not depend on previous notebook runtime variables.
It reloads data/model, writes `pipeline.py`, calibrates model probabilities, runs pipeline demos,
evaluates threshold sensitivity, and saves outputs to Google Drive.

In [ ]:
# Cell 6 - Self-sustained setup: reload eval_df + best mitigated model + inference-only trainer
import os
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

os.environ['WANDB_DISABLED'] = 'true'

if not os.path.isdir(BEST_MODEL_DIR):
    raise RuntimeError('best_mitigated_model not found at: ' + BEST_MODEL_DIR)

all_cols = list(pd.read_csv(DATA_PATH, nrows=0).columns)
base_cols = ['comment_text', 'toxic', 'black', 'white', 'muslim', 'jewish']
missing_base = [c for c in base_cols if c not in all_cols]
if missing_base:
    raise ValueError('Missing required columns: ' + str(missing_base))

lgbtq_candidates = [
    'lgbtq',
    'homosexual_gay_or_lesbian',
    'bisexual',
    'transgender',
    'other_sexual_orientation',
]
present_lgbtq_cols = [c for c in lgbtq_candidates if c in all_cols and c != 'lgbtq']

if 'lgbtq' in all_cols:
    usecols = base_cols + ['lgbtq']
    df = pd.read_csv(DATA_PATH, usecols=usecols)
else:
    if not present_lgbtq_cols:
        raise ValueError('No lgbtq or fallback columns found in dataset.')
    usecols = base_cols + present_lgbtq_cols
    df = pd.read_csv(DATA_PATH, usecols=usecols)
    df['lgbtq'] = df[present_lgbtq_cols].max(axis=1)

df = df[['comment_text', 'toxic', 'black', 'white', 'muslim', 'jewish', 'lgbtq']].copy()
df = df.dropna(subset=['comment_text']).copy()
df['toxic_label'] = (df['toxic'] >= 0.5).astype(int)

_, eval_df = train_test_split(
    df,
    train_size=100_000,
    test_size=20_000,
    stratify=df['toxic_label'],
    random_state=42,
)
eval_df = eval_df.reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR)

class ToxicityDataset(Dataset):
    def __init__(self, data_frame):
        self.df = data_frame.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(
            str(row['comment_text']),
            max_length=128,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['toxic_label']), dtype=torch.long),
        }

eval_dataset = ToxicityDataset(eval_df)

inference_args = TrainingArguments(
    output_dir='./tmp_part5_inference',
    per_device_eval_batch_size=64,
    dataloader_drop_last=False,
    report_to='none',
)
trainer = Trainer(model=model, args=inference_args)

print('Loaded eval_df shape:', eval_df.shape)
print('Loaded best mitigated model from:', BEST_MODEL_DIR)
print('Inference-only Trainer ready.')

In [ ]:
%%writefile pipeline.py
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Any

import numpy as np
import torch

BLOCKLIST: dict[str, list[re.Pattern[str]]] = {
    "direct_threat": [
        re.compile(r"\b(?:i\s*(?:will|'ll)|we\s*(?:will|'ll))\s+(?:kill|murder|shoot|stab|hurt)\s+you\b", re.IGNORECASE),
        re.compile(r"\b(?:kill|murder|shoot|stab|hurt)\s+you\b", re.IGNORECASE),
        re.compile(r"\byou\s+(?:deserve|need)\s+to\s+(?:die|be\s+killed|be\s+shot|be\s+stabbed|be\s+hurt)\b", re.IGNORECASE),
        re.compile(r"\b(?:going\s+to|gonna)\s+(?:kill|murder|shoot|stab|hurt)\s+you\b", re.IGNORECASE),
        re.compile(r"\b(?:kill|murder|shoot|stab|hurt)\s+your\s+family\b", re.IGNORECASE),
    ],
    "self_harm_directed": [
        re.compile(r"\b(?:kill|hurt|cut|hang)\s+yourself\b", re.IGNORECASE),
        re.compile(r"\byou\s+should\s+(?:die|kill\s+yourself|hurt\s+yourself)\b", re.IGNORECASE),
        re.compile(r"\bgo\s+(?:die|hurt\s+yourself|jump\s+off\s+a\s+bridge)\b", re.IGNORECASE),
        re.compile(r"\b(?:end|take)\s+your\s+life\b", re.IGNORECASE),
    ],
    "doxxing_stalking": [
        re.compile(r"\bi\s+(?:know|found)\s+where\s+you\s+live\b", re.IGNORECASE),
        re.compile(r"\bi\s+(?:have|got)\s+your\s+(?:address|phone\s*number)\b", re.IGNORECASE),
        re.compile(r"\b(?:we\s+are|we're)\s+coming\s+to\s+your\s+(?:house|home)\b", re.IGNORECASE),
        re.compile(r"\b(?:i\s+am|i'm)\s+watching\s+you\b", re.IGNORECASE),
    ],
    "dehumanization": [
        re.compile(r"\b(?:not|isn't|aren't|are\s+not)\s+(?:human|people|person)\b", re.IGNORECASE),
        re.compile(r"\b(?:these|those)\s+(?:human|people|person)\s+are\s+(?:animals|vermin|trash)\b", re.IGNORECASE),
        re.compile(r"\b(?:human|people|person)\s+like\s+you\s+are\s+(?:filth|garbage|subhuman)\b", re.IGNORECASE),
        re.compile(r"\b(?:exterminate|remove)\s+(?:that|these|those)?\s*(?:human|people|person)\b", re.IGNORECASE),
    ],
    "coordinated_harassment": [
        re.compile(r"(?=.*\beveryone\b)(?=.*\b(report|dogpile|harass)\b)(?=.*\bthem\b)", re.IGNORECASE),
        re.compile(r"\blet'?s\s+all\s+(?:report|mass\s*report|harass|spam)\s+\w+", re.IGNORECASE),
        re.compile(r"\bjoin\s+me\s+to\s+(?:harass|target|attack)\s+\w+", re.IGNORECASE),
    ],
}

def input_filter(text: str) -> dict[str, Any] | None:
    for category, patterns in BLOCKLIST.items():
        for pattern in patterns:
            if pattern.search(text):
                return {
                    "decision": "block",
                    "layer": "input_filter",
                    "category": category,
                    "confidence": 1.0,
                }
    return None

def batch_predict_probs(model: Any, tokenizer: Any, texts: list[str], batch_size: int = 64, max_length: int = 128, device: str | None = None) -> np.ndarray:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.to(device)
    model.eval()

    probs_out: list[np.ndarray] = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = [str(t) for t in texts[i:i + batch_size]]
            encoded = tokenizer(batch_texts, max_length=max_length, truncation=True, padding=True, return_tensors="pt")
            encoded = {k: v.to(device) for k, v in encoded.items()}
            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            probs_out.append(probs.detach().cpu().numpy())

    if not probs_out:
        return np.array([], dtype=float)
    return np.concatenate(probs_out).astype(float)

@dataclass
class ModerationPipeline:
    model: Any
    tokenizer: Any
    calibrator: Any
    block_threshold: float = 0.6
    allow_threshold: float = 0.4

    def _raw_prob(self, text: str) -> float:
        probs = batch_predict_probs(self.model, self.tokenizer, [text], batch_size=1)
        return float(probs[0])

    def _calibrated_prob(self, raw_prob: float) -> float:
        calibrated = self.calibrator.predict(np.array([raw_prob], dtype=float))
        return float(np.clip(calibrated[0], 0.0, 1.0))

    def predict(self, text: str) -> dict[str, Any]:
        layer1 = input_filter(text)
        if layer1 is not None:
            return layer1

        raw_prob = self._raw_prob(text)
        calibrated_prob = self._calibrated_prob(raw_prob)

        if calibrated_prob >= self.block_threshold:
            return {
                "decision": "block",
                "layer": "model_block",
                "category": "model_toxicity",
                "confidence": calibrated_prob,
            }

        if calibrated_prob <= self.allow_threshold:
            return {
                "decision": "allow",
                "layer": "model_allow",
                "category": "safe",
                "confidence": 1.0 - calibrated_prob,
            }

        return {
            "decision": "review",
            "layer": "human_review",
            "category": "uncertain",
            "confidence": 1.0 - abs(calibrated_prob - 0.5) * 2.0,
        }

In [ ]:
# Cell 8 - Calibration using trainer.predict + curve plots
import importlib
import shutil

import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

import pipeline
importlib.reload(pipeline)

pred_out = trainer.predict(eval_dataset)
logits = np.asarray(pred_out.predictions)
logits = logits - logits.max(axis=-1, keepdims=True)
probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
raw_probs = probs[:, 1].astype(float)
y_true = eval_df['toxic_label'].astype(int).to_numpy()

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(raw_probs, y_true)
calibrated_probs = calibrator.predict(raw_probs)

frac_pos_before, mean_pred_before = calibration_curve(y_true, raw_probs, n_bins=10, strategy='uniform')
frac_pos_after, mean_pred_after = calibration_curve(y_true, calibrated_probs, n_bins=10, strategy='uniform')

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
axes[0].plot(mean_pred_before, frac_pos_before, marker='o', label='Raw model')
axes[0].set_title('Before calibration')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
axes[1].plot(mean_pred_after, frac_pos_after, marker='o', color='tab:orange', label='Isotonic calibrated')
axes[1].set_title('After calibration')
axes[1].set_xlabel('Mean predicted probability')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
calibration_plot_path = os.path.join(DRIVE_OUT_DIR, 'calibration_before_after.png')
plt.savefig(calibration_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', calibration_plot_path)

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
shutil.copy2('pipeline.py', os.path.join(DRIVE_OUT_DIR, 'pipeline.py'))
print('Saved pipeline.py to Drive.')

moderation_pipeline = pipeline.ModerationPipeline(
    model=model,
    tokenizer=tokenizer,
    calibrator=calibrator,
    block_threshold=0.6,
    allow_threshold=0.4,
)

In [ ]:
# Cell 9 - Smoke test on 5 cases covering all 3 layers
smoke_cases = [
    'I know where you live and I am watching you.',
    'You should kill yourself right now.',
    'This comment is likely toxic and hateful and should be removed.',
    'I disagree with your opinion but this is not abuse.',
    'This is suspicious and could be either sarcasm or harassment.',
]

print('===== SMOKE TEST RESULTS =====')
for i, text in enumerate(smoke_cases, 1):
    out = moderation_pipeline.predict(text)
    print(f'Case {i}: {text}')
    print(' ->', out)
    print('-' * 80)

In [ ]:
# Cell 10 - Demonstration on 1000 samples
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

demo_df = eval_df.sample(n=1000, random_state=42).reset_index(drop=True).copy()
demo_df['pipeline_out'] = demo_df['comment_text'].astype(str).apply(moderation_pipeline.predict)
demo_df['decision'] = demo_df['pipeline_out'].apply(lambda x: x['decision'])
demo_df['layer'] = demo_df['pipeline_out'].apply(lambda x: x['layer'])
demo_df['category'] = demo_df['pipeline_out'].apply(lambda x: x['category'])
demo_df['confidence'] = demo_df['pipeline_out'].apply(lambda x: float(x['confidence']))

layer_order = ['input_filter', 'model_block', 'model_allow', 'human_review']
layer_counts = demo_df['layer'].value_counts().reindex(layer_order, fill_value=0)

plt.figure(figsize=(7, 7))
plt.pie(layer_counts.values, labels=layer_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Pipeline layer distribution (n=1000)')
plt.tight_layout()
pie_path = os.path.join(DRIVE_OUT_DIR, 'layer_distribution_pie.png')
plt.savefig(pie_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', pie_path)

auto_mask = demo_df['layer'] != 'human_review'
auto_df = demo_df.loc[auto_mask].copy()
auto_df['pred_label'] = (auto_df['decision'] == 'block').astype(int)

if len(auto_df) > 0:
    auto_f1 = float(f1_score(auto_df['toxic_label'], auto_df['pred_label'], zero_division=0))
    auto_precision = float(precision_score(auto_df['toxic_label'], auto_df['pred_label'], zero_division=0))
    auto_recall = float(recall_score(auto_df['toxic_label'], auto_df['pred_label'], zero_division=0))
else:
    auto_f1 = 0.0
    auto_precision = 0.0
    auto_recall = 0.0

print('===== AUTO-ACTIONED SUBSET METRICS =====')
print('Auto subset size:', len(auto_df))
print(f'F1: {auto_f1:.4f}')
print(f'Precision: {auto_precision:.4f}')
print(f'Recall: {auto_recall:.4f}')

review_df = demo_df.loc[demo_df['layer'] == 'human_review'].copy()
review_breakdown = review_df['toxic_label'].value_counts().reindex([1, 0], fill_value=0)
review_total = max(len(review_df), 1)
print('===== REVIEW QUEUE BREAKDOWN =====')
print('Review queue size:', len(review_df))
print('Toxic:', int(review_breakdown[1]), f'({review_breakdown[1] / review_total * 100:.2f}%)')
print('Non-toxic:', int(review_breakdown[0]), f'({review_breakdown[0] / review_total * 100:.2f}%)')

blocklist_hits = demo_df.loc[demo_df['layer'] == 'input_filter', 'category'].value_counts()
if blocklist_hits.empty:
    blocklist_hits = pd.Series({'none': 0})

plt.figure(figsize=(8, 5))
plt.barh(blocklist_hits.index.astype(str), blocklist_hits.values, color='tab:red', alpha=0.85)
plt.title('Blocklist category hit counts (n=1000)')
plt.xlabel('Count')
plt.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
blocklist_plot_path = os.path.join(DRIVE_OUT_DIR, 'blocklist_category_hits.png')
plt.savefig(blocklist_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', blocklist_plot_path)

In [ ]:
# Cell 11 - Threshold sensitivity analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score

bands = [
    ('narrow(0.45-0.55)', 0.45, 0.55),
    ('current(0.40-0.60)', 0.40, 0.60),
    ('wide(0.30-0.70)', 0.30, 0.70),
]

texts_1000 = demo_df['comment_text'].astype(str).tolist()
y_1000 = demo_df['toxic_label'].astype(int).to_numpy()

raw_probs_1000 = pipeline.batch_predict_probs(model, tokenizer, texts_1000, batch_size=64)
cal_probs_1000 = calibrator.predict(raw_probs_1000)

input_hits = [pipeline.input_filter(t) for t in texts_1000]

rows = []
for band_name, allow_th, block_th in bands:
    decisions = []
    pred_labels = []
    layers = []

    for i, hit in enumerate(input_hits):
        if hit is not None:
            layers.append('input_filter')
            decisions.append('block')
            pred_labels.append(1)
            continue

        p = float(cal_probs_1000[i])
        if p >= block_th:
            layers.append('model_block')
            decisions.append('block')
            pred_labels.append(1)
        elif p <= allow_th:
            layers.append('model_allow')
            decisions.append('allow')
            pred_labels.append(0)
        else:
            layers.append('human_review')
            decisions.append('review')
            pred_labels.append(-1)

    layers = np.asarray(layers)
    pred_labels = np.asarray(pred_labels)

    review_mask = layers == 'human_review'
    auto_mask = ~review_mask
    review_pct = float(review_mask.mean() * 100.0)

    if auto_mask.sum() > 0:
        auto_f1 = float(f1_score(y_1000[auto_mask], pred_labels[auto_mask], zero_division=0))
        auto_precision = float(precision_score(y_1000[auto_mask], pred_labels[auto_mask], zero_division=0))
        auto_recall = float(recall_score(y_1000[auto_mask], pred_labels[auto_mask], zero_division=0))
    else:
        auto_f1 = 0.0
        auto_precision = 0.0
        auto_recall = 0.0

    rows.append({
        'Band': band_name,
        'allow_th': allow_th,
        'block_th': block_th,
        'review_pct': review_pct,
        'auto_f1': auto_f1,
        'auto_precision': auto_precision,
        'auto_recall': auto_recall,
    })

sensitivity_df = pd.DataFrame(rows)

print('===== THRESHOLD SENSITIVITY TABLE =====')
print(sensitivity_df[['Band', 'review_pct', 'auto_f1', 'auto_precision', 'auto_recall']].round(4).to_string(index=False))

x = np.arange(len(sensitivity_df))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - 1.5 * width, sensitivity_df['review_pct'] / 100.0, width, label='review_pct', color='#5DA5DA')
ax.bar(x - 0.5 * width, sensitivity_df['auto_f1'], width, label='auto_f1', color='#60BD68')
ax.bar(x + 0.5 * width, sensitivity_df['auto_precision'], width, label='auto_precision', color='#F17CB0')
ax.bar(x + 1.5 * width, sensitivity_df['auto_recall'], width, label='auto_recall', color='#B2912F')

ax.set_xticks(x)
ax.set_xticklabels(sensitivity_df['Band'], rotation=10)
ax.set_ylim(0, 1)
ax.set_ylabel('Value (0-1 scale)')
ax.set_title('Threshold band sensitivity: review vs auto metrics')
ax.grid(True, axis='y', alpha=0.3)
ax.legend(loc='best')

plt.tight_layout()
sensitivity_plot_path = os.path.join(DRIVE_OUT_DIR, 'threshold_sensitivity.png')
plt.savefig(sensitivity_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', sensitivity_plot_path)

sensitivity_csv = os.path.join(DRIVE_OUT_DIR, 'threshold_sensitivity_table.csv')
sensitivity_df.to_csv(sensitivity_csv, index=False)
print('Saved:', sensitivity_csv)

In [ ]:
# Cell 12 - Markdown analysis and final band recommendation
from IPython.display import Markdown, display

rank_df = sensitivity_df.copy()
rank_df['safety_score'] = rank_df['auto_recall']
rank_df['ux_score'] = 1.0 - (rank_df['review_pct'] / 100.0)
rank_df['ops_score'] = 1.0 - (rank_df['review_pct'] / 100.0)
rank_df['combined_score'] = 0.5 * rank_df['safety_score'] + 0.3 * rank_df['ux_score'] + 0.2 * rank_df['auto_precision']

best_idx = rank_df['combined_score'].idxmax()
recommended_band = rank_df.loc[best_idx, 'Band']

def _row_text(band_name):
    r = sensitivity_df[sensitivity_df['Band'] == band_name].iloc[0]
    return f"- {band_name}: review={r['review_pct']:.2f}%, auto F1={r['auto_f1']:.4f}, precision={r['auto_precision']:.4f}, recall={r['auto_recall']:.4f}"

analysis_md = f"""
## Band Selection Analysis

### Is 0.4-0.6 the right band?
The current band is competitive, but the final decision should balance safety, user experience, and ops cost.

### Measured results
{_row_text('narrow(0.45-0.55)')}
{_row_text('current(0.40-0.60)')}
{_row_text('wide(0.30-0.70)')}

### Recommendation
Recommended final band: **{recommended_band}**.

Justification:
- **Safety:** prioritized using auto recall (fewer toxic misses in auto actions).
- **User experience:** lower review% means faster final decisions for users.
- **Operational cost:** review% directly drives human moderation workload.

If policy is safety-max-first, prefer the highest-recall band even if review load rises.
If moderation capacity is constrained, prefer the lowest review% band with acceptable recall.
"""

display(Markdown(analysis_md))